# Neteja dels datasets final i feature enineering

Aquest notebook té com objectiu netejar les dades i crear variables per enriquir les dades i obtenir millors resultats. Es tractaran les dadess seguint l'enfocament de la ingesta, per tipologia de les dades. 

Finalment, es combinaran les dades de les diferents topologies per obtenir 3 datasets finals lleestos per al modelatge:
- **Dataset de 2015**
- **Dataset de 2023**
- **Dataset amb deltes**


# Llibreries

In [2]:
import pandas as pd
import numpy as np
import json
import os
import sys
from pathlib import Path

# Visualització
import matplotlib.pyplot as plt
import seaborn as sns

# Config vis
sns.set_theme()

# Funcions
cwd = os.getcwd()
parent = os.path.abspath(os.path.join(cwd, os.pardir))
sys.path.insert(0, parent)
from src.utils import neteja_noms_columnes

# Funcions auxiliars

In [66]:
def resum_estructura(df, nom_dataset):
    """
    Funció per estudiar l'estructura de les dades d' un dataset i detectar valors nuls, infinits, i valors únics.
    """
    resum = pd.DataFrame({
        'tipus': df.dtypes.astype(str),
        'nuls': df.isna().sum(),
        'infinits': np.isinf(df.select_dtypes(include=[np.number])).sum().reindex(df.columns, fill_value=0)
    })
    resum.index.name = f'columna ({nom_dataset})'
    return resum


def calcul_deltes_pct(df, variables):
    df_mod = df.copy()
    for v in variables:
        df_mod[f"delta_{v}"] = df_mod[f"{v}_2023"] - df_mod[f"{v}_2015"]
    return df_mod


def calcul_deltes_absouts(df, variables):
    df_mod = df.copy()
    for v in variables:
        df_mod[f"delta_{v}"] = np.where(df_mod[f"{v}_2015"] == 0, 0, (df_mod[f"{v}_2023"] - df_mod[f"{v}_2015"]) / df_mod[f"{v}_2015"])
    return df_mod

# Dimensions
Carrega del dataset que conté totes les dimensions de les dades

In [3]:
# Carrega de dimensions
dimensions = pd.read_csv("../data/dimensions/pad_dimensions.csv")
dim_barris = pd.read_csv("../data/dimensions/BarcelonaCiutat_Barris.csv")

# Dades Demogràfiques

### Neteja de dades
En aquesta secció identificarem valors erronis com nuls o duplicats

In [14]:
poblacio_23_raw = pd.read_csv("../data/processed/df_demografia_23.csv")
resum_estructura(poblacio_23_raw, "poblacio_23")

,tipus,nuls,infinits
columna (poblacio_23),,,
codi_barri,int64,0,0
poblacio_total,int64,0,0
espanya,int64,0,0
no_consta_x,int64,0,0
resta_de_la_uni_europea,int64,0,0
resta_del_mn,int64,0,0
amrica_central,int64,0,0
amrica_del_nord,int64,0,0
amrica_del_sud,int64,0,0


**Observacions:**
- No s' observen valors nuls

In [15]:
poblacio_15_raw = pd.read_csv("../data/processed/df_demografia_15.csv")
resum_estructura(poblacio_15_raw, "poblacio_15")

,tipus,nuls,infinits
columna (poblacio_15),,,
codi_barri,int64,0,0
poblacio_total,int64,0,0
espanya,int64,0,0
no_consta_x,int64,0,0
resta_de_la_uni_europea,int64,0,0
resta_del_mn,int64,0,0
amrica_central,int64,0,0
amrica_del_nord,int64,0,0
amrica_del_sud,int64,0,0


**Observacions:**
- No s' observen valors nuls ne les dades 

### Feature Engineering
En aquest apartat enriquirem els datasets amb algunes variables porcentuals que ens ajudaran a modelar segons el nostre objectiu.
- Població Total
- % població estrangera
- % estragners_occidentals (Amèrica del Nord i Europa) - Article Lopez Gay o Cocola Gant?
- % pct_universitaris/estudis_superiors
- % pct_sense_estudis


In [9]:
# Apilem primer per aplicar transformacions
poblacio = pd.concat([poblacio_23_raw, poblacio_15_raw], keys=[2023, 2015]).reset_index(names=["periode", "index"]).drop(columns=["index"])
poblacio.head()

,periode,codi_barri,poblacio_total,espanya,no_consta_x,resta_de_la_uni_europea,resta_del_mn,amrica_central,amrica_del_nord,amrica_del_sud,...,joves_espanya,joves_no_consta,joves_resta_de_la_uni_europea,joves_resta_del_mn,joves_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_pob_espanyola,pct_joves,pct_universitaris
0,2023,1,45671,22396,56,4563,18693,470,297,2888,...,3253,29,2362,11771,17415,0.509623,0.106413,0.490377,0.381314,0.241860
1,2023,2,24460,8385,32,3281,12784,453,306,2203,...,1435,16,1667,9196,12314,0.657195,0.146648,0.342805,0.503434,0.278250
2,2023,3,14274,8264,4,2619,3390,184,129,1248,...,1465,4,1369,2992,5830,0.421045,0.192518,0.578955,0.408435,0.305240
3,2023,4,22041,11764,16,4284,5989,313,390,1736,...,2016,16,2189,5049,9270,0.466267,0.212059,0.533733,0.420580,0.406288
4,2023,5,34403,23354,16,3445,7600,608,151,3088,...,3851,20,1270,6043,11184,0.321164,0.104526,0.678836,0.325088,0.377961


In [10]:
# Calcul de percentatges
poblacio["pct_pob_estrangera"] = 1 -(poblacio["espanya"] / poblacio["poblacio_total"])
poblacio["pct_pob_estrangera_occidental"] = (poblacio["resta_de_la_uni_europea"] + poblacio["amrica_del_nord"]) / poblacio["poblacio_total"]
poblacio["pct_joves"] = poblacio["joves_total"] / poblacio["poblacio_total"]
poblacio["pct_universitaris"] = (poblacio["estudis_superiors_espanya"] + poblacio["estudis_superiors_ue"] + poblacio["estudis_superiors_mon"]) / poblacio["poblacio_total"]

In [11]:
# Seleccionem conjunt d' atributs finals
poblacio = poblacio[["codi_barri", "periode", "poblacio_total", "pct_pob_estrangera", "pct_pob_estrangera_occidental", "pct_joves", "pct_universitaris"]]
poblacio.head()

,codi_barri,periode,poblacio_total,pct_pob_estrangera,pct_pob_estrangera_occidental,pct_joves,pct_universitaris
0,1,2023,45671,0.509623,0.106413,0.381314,0.241860
1,2,2023,24460,0.657195,0.146648,0.503434,0.278250
2,3,2023,14274,0.421045,0.192518,0.408435,0.305240
3,4,2023,22041,0.466267,0.212059,0.420580,0.406288
4,5,2023,34403,0.321164,0.104526,0.325088,0.377961


Abans de separar el conjunt de dades, comprovem que no hagin aparegut valors erronis

In [16]:
resum_estructura(poblacio, "poblacio")

,tipus,nuls,infinits
columna (poblacio),,,
codi_barri,int64,0,0
periode,int64,0,0
poblacio_total,int64,0,0
pct_pob_estrangera,float64,0,0
pct_pob_estrangera_occidental,float64,0,0
pct_joves,float64,0,0
pct_universitaris,float64,0,0


**Observacions:**
- Les dades no presenten errors i el tipus es correcte

In [17]:
# Separem les dades per 2023 i 2015 per posteriorment crear un conjunt de modelling per als dos
poblacio_15 = poblacio[poblacio["periode"] == 2015].drop(columns= ["periode"])
poblacio_23 = poblacio[poblacio["periode"] == 2023].drop(columns= ["periode"])

In [23]:
poblacio_pivot = poblacio.pivot(index="codi_barri", columns="periode")
poblacio_pivot.head()

columna (poblacio) poblacio_total        pct_pob_estrangera            \
periode                      2015   2023               2015      2023   
codi_barri                                                              
1                           47150  45671           0.480424  0.509623   
2                           15514  24460           0.419428  0.657195   
3                           15037  14274           0.312961  0.421045   
4                           22468  22041           0.392736  0.466267   
5                           31548  34403           0.194339  0.321164   

columna (poblacio) pct_pob_estrangera_occidental           pct_joves  \
periode                                     2015      2023      2015   
codi_barri                                                             
1                                       0.095949  0.106413  0.394040   
2                                       0.209037  0.146648  0.428129   
3                                       0.151160  0.192518  0.373213   
4                                       0.194632  0.212059  0.403819   
5                                       0.076455  0.104526  0.296215   

columna (poblacio)           pct_universitaris            
periode                 2023              2015      2023  
codi_barri                                                
1                   0.381314          0.184963  0.241860  
2                   0.503434          0.333570  0.278250  
3                   0.408435          0.214737  0.305240  
4                   0.420580          0.323393  0.406288  
5                   0.325088          0.313966  0.377961

In [24]:
poblacio_pivot.columns = [f"{col}_{year}" for col, year in poblacio_pivot.columns]
poblacio_pivot = poblacio_pivot.reset_index()
poblacio_pivot.head()

,codi_barri,poblacio_total_2015,poblacio_total_2023,pct_pob_estrangera_2015,pct_pob_estrangera_2023,pct_pob_estrangera_occidental_2015,pct_pob_estrangera_occidental_2023,pct_joves_2015,pct_joves_2023,pct_universitaris_2015,pct_universitaris_2023
0,1,47150,45671,0.480424,0.509623,0.095949,0.106413,0.394040,0.381314,0.184963,0.241860
1,2,15514,24460,0.419428,0.657195,0.209037,0.146648,0.428129,0.503434,0.333570,0.278250
2,3,15037,14274,0.312961,0.421045,0.151160,0.192518,0.373213,0.408435,0.214737,0.305240
3,4,22468,22041,0.392736,0.466267,0.194632,0.212059,0.403819,0.420580,0.323393,0.406288
4,5,31548,34403,0.194339,0.321164,0.076455,0.104526,0.296215,0.325088,0.313966,0.377961


In [25]:
# Deltas
vars_delta_pct = [
    "pct_pob_estrangera",
    "pct_pob_estrangera_occidental",
    "pct_joves",
    "pct_universitaris"
]


vars_delta_abs = ["poblacio_total"]

poblacio_metriques_pct = calcul_deltes_pct(poblacio_pivot, vars_delta_pct)
poblacio_metriques_abs = calcul_deltes_absouts(poblacio_metriques_pct, vars_delta_abs)

poblacio_metriques_abs.head()

,codi_barri,poblacio_total_2015,poblacio_total_2023,pct_pob_estrangera_2015,pct_pob_estrangera_2023,pct_pob_estrangera_occidental_2015,pct_pob_estrangera_occidental_2023,pct_joves_2015,pct_joves_2023,pct_universitaris_2015,pct_universitaris_2023,delta_pct_pob_estrangera,delta_pct_pob_estrangera_occidental,delta_pct_joves,delta_pct_universitaris,delta_poblacio_total
0,1,47150,45671,0.480424,0.509623,0.095949,0.106413,0.394040,0.381314,0.184963,0.241860,0.029199,0.010464,-0.012726,0.056897,-0.031368
1,2,15514,24460,0.419428,0.657195,0.209037,0.146648,0.428129,0.503434,0.333570,0.278250,0.237768,-0.062389,0.075305,-0.055319,0.576640
2,3,15037,14274,0.312961,0.421045,0.151160,0.192518,0.373213,0.408435,0.214737,0.305240,0.108084,0.041357,0.035222,0.090503,-0.050742
3,4,22468,22041,0.392736,0.466267,0.194632,0.212059,0.403819,0.420580,0.323393,0.406288,0.073531,0.017427,0.016761,0.082895,-0.019005
4,5,31548,34403,0.194339,0.321164,0.076455,0.104526,0.296215,0.325088,0.313966,0.377961,0.126825,0.028071,0.028873,0.063995,0.090497


In [26]:
# Ens quedem amb els deltas per al dataset final de poblacio
poblacio_final = poblacio_metriques_abs[[col for col in poblacio_metriques_abs.columns if col.startswith("delta") or col == "codi_barri"]]
poblacio_final.head()

,codi_barri,delta_pct_pob_estrangera,delta_pct_pob_estrangera_occidental,delta_pct_joves,delta_pct_universitaris,delta_poblacio_total
0,1,0.029199,0.010464,-0.012726,0.056897,-0.031368
1,2,0.237768,-0.062389,0.075305,-0.055319,0.576640
2,3,0.108084,0.041357,0.035222,0.090503,-0.050742
3,4,0.073531,0.017427,0.016761,0.082895,-0.019005
4,5,0.126825,0.028071,0.028873,0.063995,0.090497


In [28]:
resum_estructura(poblacio_final, "poblacio_final")

,tipus,nuls,infinits
columna (poblacio_final),,,
codi_barri,int64,0,0
delta_pct_pob_estrangera,float64,0,0
delta_pct_pob_estrangera_occidental,float64,0,0
delta_pct_joves,float64,0,0
delta_pct_universitaris,float64,0,0
delta_poblacio_total,float64,0,0


**Observacions:**
- No s' observen valors nuls ni infinits

# Dades Econòmiques
Incorporem dades econòmiques:
- Renda neta mitja per persona.
- Índex de gini

### neteja i validacions

In [32]:
econs_15 = pd.read_csv("../data/processed/df_economiques_15.csv")
resum_estructura(econs_15, "econs_15")

,tipus,nuls,infinits
columna (econs_15),,,
codi_barri,int64,0,0
import_euros,float64,0,0
index_gini,float64,0,0


In [34]:
econs_23 = pd.read_csv("../data/processed/df_economiques_23.csv")
resum_estructura(econs_23, "econs_23")

,tipus,nuls,infinits
columna (econs_23),,,
codi_barri,int64,0,0
import_euros,float64,0,0
index_gini,float64,0,0


### Feature Engineering
En aquest apartat obtindrem els deltes per al conjunt final de dades.

In [35]:
# Combinem
econs = dim_barris[["codi_barri"]].copy()
econs_merged = econs.merge(econs_23, on="codi_barri", how="left")\
    .merge(econs_15, on = "codi_barri", suffixes=["_2023", "_2015"])

econs_merged.head()

,codi_barri,import_euros_2023,index_gini_2023,import_euros_2015,index_gini_2015
0,1,11917.476190,33.795238,8507.809524,36.338095
1,2,16013.222222,39.266667,11447.444444,41.087500
2,3,15359.818182,32.809091,10818.818182,34.954545
3,4,16621.153846,36.769231,11689.000000,39.515385
4,5,20418.800000,31.890000,15441.500000,34.020000


In [38]:
# Calculem deltes
econ_vars_delta_abs = ["import_euros", "index_gini"]
econs_final = calcul_deltes_absouts(econs_merged, econ_vars_delta_abs)

econs_final.head()

,codi_barri,import_euros_2023,index_gini_2023,import_euros_2015,index_gini_2015,delta_import_euros,delta_index_gini
0,1,11917.476190,33.795238,8507.809524,36.338095,0.400769,-0.069978
1,2,16013.222222,39.266667,11447.444444,41.087500,0.398847,-0.044316
2,3,15359.818182,32.809091,10818.818182,34.954545,0.419732,-0.061378
3,4,16621.153846,36.769231,11689.000000,39.515385,0.421948,-0.069496
4,5,20418.800000,31.890000,15441.500000,34.020000,0.322333,-0.062610


In [40]:
# Ens  quedem amb els deltes només
econs_final = econs_final[[col for col in econs_final.columns if col.startswith("delta") or col == "codi_barri"]]
resum_estructura(econs_final, "econs_final")

,tipus,nuls,infinits
columna (econs_final),,,
codi_barri,int64,0,0
delta_import_euros,float64,0,0
delta_index_gini,float64,0,0


# Dades Urbanes
En aquest cas les dades són en valors absoluts. Per tal de neutralitzar l'efecte esbiaixador provocat per barris amb més població, normalitzarem les dades a un càlcul de cada 1000 habitants.

### Neteja

In [41]:
df_urbanes_23_raw = pd.read_csv("../data/processed/df_urbanes_23.csv")
resum_estructura(df_urbanes_23_raw, "urbanes_23")

,tipus,nuls,infinits
columna (urbanes_23),,,
codi_barri,int64,0,0
total_incidents,int64,0,0
locals_restauracio,int64,0,0
locals_sanitaris,int64,0,0
locals_serveis_professionals,int64,0,0


In [42]:
df_urbanes_15_raw = pd.read_csv("../data/processed/df_urbanes_15.csv")
resum_estructura(df_urbanes_15_raw, "urbanes_15")

,tipus,nuls,infinits
columna (urbanes_15),,,
codi_barri,int64,0,0
total_incidents,int64,0,0
locals_restauracio,int64,0,0
locals_sanitaris,int64,0,0
locals_serveis_professionals,int64,0,0


In [44]:
# Combinem els dos datasets
df_urbanes = pd.concat([df_urbanes_23_raw, df_urbanes_15_raw], keys= [2023, 2015]).reset_index(names=["periode", "remove"]).drop(columns=["remove"])
df_urbanes.head()

,periode,codi_barri,total_incidents,locals_restauracio,locals_sanitaris,locals_serveis_professionals
0,2023,1,748,500,8,10
1,2023,2,700,440,6,2
2,2023,3,651,177,15,5
3,2023,4,676,387,13,2
4,2023,5,638,200,23,33


In [45]:
# Obtenim dades de la població en barris per periode
df_urbanes_join = df_urbanes.merge(poblacio[["codi_barri", "periode", "poblacio_total"]], on = ["codi_barri", "periode"], how= "left")
resum_estructura(df_urbanes_join, "joined_urbanes")

,tipus,nuls,infinits
columna (joined_urbanes),,,
periode,int64,0,0
codi_barri,int64,0,0
total_incidents,int64,0,0
locals_restauracio,int64,0,0
locals_sanitaris,int64,0,0
locals_serveis_professionals,int64,0,0
poblacio_total,int64,0,0


- Info de tots els barris, no hi ha nuls

In [46]:
# Normalitzem valors
df_urbanes_metriques = df_urbanes_join.copy()
metriques = ['total_incidents', 'locals_restauracio','locals_sanitaris', 'locals_serveis_professionals']

for metrica in metriques:
    df_urbanes_metriques[f"{metrica}_1000_hab"] = df_urbanes_metriques[metrica] / df_urbanes_metriques["poblacio_total"] * 1000

df_urbanes_metriques.head()

columna (joined_urbanes),periode,codi_barri,total_incidents,locals_restauracio,locals_sanitaris,locals_serveis_professionals,poblacio_total,total_incidents_1000_hab,locals_restauracio_1000_hab,locals_sanitaris_1000_hab,locals_serveis_professionals_1000_hab
0,2023,1,748,500,8,10,45671,16.378008,10.947866,0.175166,0.218957
1,2023,2,700,440,6,2,24460,28.618152,17.988553,0.245298,0.081766
2,2023,3,651,177,15,5,14274,45.607398,12.400168,1.050862,0.350287
3,2023,4,676,387,13,2,22041,30.670115,17.558187,0.589810,0.090740
4,2023,5,638,200,23,33,34403,18.544894,5.813447,0.668546,0.959219


In [47]:
# Seleccionem les columnes que acabem de crear
df_urbanes_clean = df_urbanes_metriques[[col for col in df_urbanes_metriques.columns if col.endswith("_hab") or col in ["codi_barri", "periode"]]]
resum_estructura(df_urbanes_clean, "urbanes_clean")

,tipus,nuls,infinits
columna (urbanes_clean),,,
periode,int64,0,0
codi_barri,int64,0,0
total_incidents_1000_hab,float64,0,0
locals_restauracio_1000_hab,float64,0,0
locals_sanitaris_1000_hab,float64,0,0
locals_serveis_professionals_1000_hab,float64,0,0


In [48]:
# Separem els dos conjunts de dades
df_urbanes_23 = df_urbanes_clean[df_urbanes_clean["periode"] == 2023].drop(columns=["periode"])
df_urbanes_15 = df_urbanes_clean[df_urbanes_clean["periode"] == 2015].drop(columns=["periode"])

In [49]:
df_urbanes_pivot = df_urbanes_clean.pivot(index = "codi_barri", columns="periode")
df_urbanes_pivot.head()

columna (urbanes_clean) total_incidents_1000_hab             \
periode                                     2015       2023   
codi_barri                                                    
1                                      14.294804  16.378008   
2                                      40.286193  28.618152   
3                                      40.899116  45.607398   
4                                      26.348585  30.670115   
5                                      17.877520  18.544894   

columna (urbanes_clean) locals_restauracio_1000_hab             \
periode                                        2015       2023   
codi_barri                                                       
1                                         11.537646  10.947866   
2                                         33.324739  17.988553   
3                                         14.497573  12.400168   
4                                         16.645896  17.558187   
5                                          5.515405   5.813447   

columna (urbanes_clean) locals_sanitaris_1000_hab            \
periode                                      2015      2023   
codi_barri                                                    
1                                        0.148462  0.175166   
2                                        0.257832  0.245298   
3                                        0.532021  1.050862   
4                                        0.356062  0.589810   
5                                        0.507164  0.668546   

columna (urbanes_clean) locals_serveis_professionals_1000_hab            
periode                                                  2015      2023  
codi_barri                                                               
1                                                    0.699894  0.218957  
2                                                    0.322290  0.081766  
3                                                    0.399016  0.350287  
4                                                    0.979170  0.090740  
5                                                    1.743375  0.959219

In [50]:
# Modifiquem noms de columnes
df_urbanes_pivot.columns = [f"{col}_{year}" for col, year in df_urbanes_pivot.columns]
df_urbanes_pivot = df_urbanes_pivot.reset_index()
df_urbanes_pivot.head()

,codi_barri,total_incidents_1000_hab_2015,total_incidents_1000_hab_2023,locals_restauracio_1000_hab_2015,locals_restauracio_1000_hab_2023,locals_sanitaris_1000_hab_2015,locals_sanitaris_1000_hab_2023,locals_serveis_professionals_1000_hab_2015,locals_serveis_professionals_1000_hab_2023
0,1,14.294804,16.378008,11.537646,10.947866,0.148462,0.175166,0.699894,0.218957
1,2,40.286193,28.618152,33.324739,17.988553,0.257832,0.245298,0.322290,0.081766
2,3,40.899116,45.607398,14.497573,12.400168,0.532021,1.050862,0.399016,0.350287
3,4,26.348585,30.670115,16.645896,17.558187,0.356062,0.589810,0.979170,0.090740
4,5,17.877520,18.544894,5.515405,5.813447,0.507164,0.668546,1.743375,0.959219


In [56]:
# Calculem deltes
urbanes_deltes_abs = ['total_incidents_1000_hab','locals_restauracio_1000_hab', 'locals_sanitaris_1000_hab','locals_serveis_professionals_1000_hab']

df_urbanes_metriques = calcul_deltes_absouts(df_urbanes_pivot, urbanes_deltes_abs)

df_urbanes_metriques.head()

,codi_barri,total_incidents_1000_hab_2015,total_incidents_1000_hab_2023,locals_restauracio_1000_hab_2015,locals_restauracio_1000_hab_2023,locals_sanitaris_1000_hab_2015,locals_sanitaris_1000_hab_2023,locals_serveis_professionals_1000_hab_2015,locals_serveis_professionals_1000_hab_2023,delta_total_incidents_1000_hab,delta_locals_restauracio_1000_hab,delta_locals_sanitaris_1000_hab,delta_locals_serveis_professionals_1000_hab
0,1,14.294804,16.378008,11.537646,10.947866,0.148462,0.175166,0.699894,0.218957,0.145732,-0.051118,0.179867,-0.687156
1,2,40.286193,28.618152,33.324739,17.988553,0.257832,0.245298,0.322290,0.081766,-0.289629,-0.460204,-0.048610,-0.746296
2,3,40.899116,45.607398,14.497573,12.400168,0.532021,1.050862,0.399016,0.350287,0.115119,-0.144673,0.975226,-0.122122
3,4,26.348585,30.670115,16.645896,17.558187,0.356062,0.589810,0.979170,0.090740,0.164014,0.054806,0.656481,-0.907330
4,5,17.877520,18.544894,5.515405,5.813447,0.507164,0.668546,1.743375,0.959219,0.037330,0.054038,0.318206,-0.449792


In [57]:
# Ens quedem amb els deltas per al dataset
urbanes_final = df_urbanes_metriques[[col for col in df_urbanes_metriques.columns if col.startswith("delta") or col == "codi_barri"]]
resum_estructura(urbanes_final, "urbanes_")

,tipus,nuls,infinits
columna (urbanes_),,,
codi_barri,int64,0,0
delta_total_incidents_1000_hab,float64,0,0
delta_locals_restauracio_1000_hab,float64,0,0
delta_locals_sanitaris_1000_hab,float64,0,0
delta_locals_serveis_professionals_1000_hab,float64,0,0


**Observacions:**
- En una primera instància apareixen nuls i infinits
- Després de corretgir la fórmula ja no hi ha presència de nuls ni infinits

# Dades habitatge
Seguint el procediment utilitzat amb les dades urbanes, les dades d'habitatge amb valors absoluts es normalitzaran per evitar els efectes de variabilitat de la poblaicó entre barris

In [59]:
df_habitatges_23_raw = pd.read_csv("../data/processed/df_habitatge_23.csv")
resum_estructura(df_habitatges_23_raw, "habitatge_23")

,tipus,nuls,infinits
columna (habitatge_23),,,
codi_barri,int64,0,0
preu_mitja_m2,float64,0,0
pisos_turistics,float64,8,0


**Observacions:**
- existeixen valors de pisos turístics nuls.

In [60]:
df_habitatges_15_raw = pd.read_csv("../data/processed/df_habitatge_15.csv")
resum_estructura(df_habitatges_15_raw, "habitatge_15")

,tipus,nuls,infinits
columna (habitatge_15),,,
codi_barri,int64,0,0
preu_mitja_m2,float64,0,0
pisos_turistics,float64,10,0


**Observacions:**
- existeixen valors de pisos turístics nuls.

In [63]:
# Apilem dades
df_habitatges_concat = pd.concat([df_habitatges_23_raw, df_habitatges_15_raw], keys = [2023, 2015]).reset_index(names = ["periode", "remove"]).drop(columns=["remove"])
df_habitatges_concat.head()

,periode,codi_barri,preu_mitja_m2,pisos_turistics
0,2023,1,15.9,212.0
1,2023,2,16.4,201.0
2,2023,3,19.0,64.0
3,2023,4,17.4,171.0
4,2023,5,16.1,333.0


Explorem aquests nuls:

In [65]:
df_habitatges_concat[df_habitatges_concat["pisos_turistics"].isna()]\
    .merge(dim_barris[["codi_barri", "nom_barri"]], on = "codi_barri", how = "left")

,periode,codi_barri,preu_mitja_m2,pisos_turistics,nom_barri
0,2023,40,13.8,NaN,Montbau
1,2023,42,15.5,NaN,la Clota
2,2023,47,12.5,NaN,Can Peguera
3,2023,53,13.5,NaN,la Trinitat Nova
4,2023,54,7.7,NaN,Torre Baró
5,2023,55,10.4,NaN,Ciutat Meridiana
6,2023,56,8.0,NaN,Vallbona
7,2023,58,9.2,NaN,Baró de Viver
8,2015,40,10.1,NaN,Montbau
9,2015,41,10.1,NaN,la Vall d'Hebron


**Observacions:**
- Són barris on no hi ha registres de pisos turístics en cap dels dos anys. COnsiderarem que és un fet possible, donat que són barris menys turístics i per tant no hi ha tant interès en tenir un pis turístic en la zona.
Per aquesta lògica, no imputarem el valor amb tècniques ni mesures centrals, simplement substituirem el valor per 0, indicant que no hi ha existència de pisos turístics en aquest registres.

In [69]:
df_habitatges_concat.loc[:, "pisos_turistics"] = df_habitatges_concat["pisos_turistics"].fillna(0)
resum_estructura(df_habitatges_concat, "concat_net")

,tipus,nuls,infinits
columna (concat_net),,,
periode,int64,0,0
codi_barri,int64,0,0
preu_mitja_m2,float64,0,0
pisos_turistics,float64,0,0


**Observacions:**
- nuls imputats

In [70]:
 # Portem informació de les poblacions en cada barri i periode
df_habitatges_join = df_habitatges_concat.merge(poblacio[["codi_barri", "periode", "poblacio_total"]], on = ["codi_barri", "periode"], how= "left")
df_habitatges_join.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   periode          146 non-null    int64  
 1   codi_barri       146 non-null    int64  
 2   preu_mitja_m2    146 non-null    float64
 3   pisos_turistics  146 non-null    float64
 4   poblacio_total   146 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 5.8 KB


In [71]:
# Creem metrica i comprovem
df_habitatges_metriques = df_habitatges_join.copy()
df_habitatges_metriques["pisos_turistics_1000_hab"] = df_habitatges_metriques["pisos_turistics"] / df_habitatges_metriques["poblacio_total"] * 1000
df_habitatges_metriques.head()

,periode,codi_barri,preu_mitja_m2,pisos_turistics,poblacio_total,pisos_turistics_1000_hab
0,2023,1,15.9,212.0,45671,4.641895
1,2023,2,16.4,201.0,24460,8.217498
2,2023,3,19.0,64.0,14274,4.483677
3,2023,4,17.4,171.0,22041,7.758269
4,2023,5,16.1,333.0,34403,9.679388


In [73]:
# Seleccionem les columnes que acabem de crear
df_habitatge_clean = df_habitatges_metriques[[col for col in df_habitatges_metriques.columns if col not in ["pisos_turistics", "poblacio_total"]]]
resum_estructura(df_habitatge_clean, "habitatge_clean")

,tipus,nuls,infinits
columna (habitatge_clean),,,
periode,int64,0,0
codi_barri,int64,0,0
preu_mitja_m2,float64,0,0
pisos_turistics_1000_hab,float64,0,0


In [74]:
# Separem els dos conjunts de dades
df_habitatge_23 = df_habitatge_clean[df_habitatge_clean["periode"] == 2023].drop(columns=["periode"])
df_habitatge_15 = df_habitatge_clean[df_habitatge_clean["periode"] == 2015].drop(columns=["periode"])

In [75]:
df_habitatges_pivot = df_habitatge_clean.pivot(index = "codi_barri", columns="periode")
df_habitatges_pivot.head()

columna (habitatge_clean) preu_mitja_m2       pisos_turistics_1000_hab  \
periode                            2015  2023                     2015   
codi_barri                                                               
1                                  11.1  15.9                 3.817603   
2                                  11.2  16.4                11.860255   
3                                  16.3  19.0                 4.588681   
4                                  12.7  17.4                 7.610824   
5                                  10.9  16.1                10.872322   

columna (habitatge_clean)            
periode                        2023  
codi_barri                           
1                          4.641895  
2                          8.217498  
3                          4.483677  
4                          7.758269  
5                          9.679388

In [76]:
# Combinem noms de les columnes
df_habitatges_pivot.columns = [f"{col}_{year}" for col, year in df_habitatges_pivot.columns]
df_habitatges_pivot = df_habitatges_pivot.reset_index()
df_habitatges_pivot.head()

,codi_barri,preu_mitja_m2_2015,preu_mitja_m2_2023,pisos_turistics_1000_hab_2015,pisos_turistics_1000_hab_2023
0,1,11.1,15.9,3.817603,4.641895
1,2,11.2,16.4,11.860255,8.217498
2,3,16.3,19.0,4.588681,4.483677
3,4,12.7,17.4,7.610824,7.758269
4,5,10.9,16.1,10.872322,9.679388


In [78]:
hab_deltes_abs = ["preu_mitja_m2", "pisos_turistics_1000_hab"]

df_habitatges_metriques = calcul_deltes_absouts(df_habitatges_pivot, hab_deltes_abs)
df_habitatges_metriques.head()

,codi_barri,preu_mitja_m2_2015,preu_mitja_m2_2023,pisos_turistics_1000_hab_2015,pisos_turistics_1000_hab_2023,delta_preu_mitja_m2,delta_pisos_turistics_1000_hab
0,1,11.1,15.9,3.817603,4.641895,0.432432,0.215919
1,2,11.2,16.4,11.860255,8.217498,0.464286,-0.307140
2,3,16.3,19.0,4.588681,4.483677,0.165644,-0.022883
3,4,12.7,17.4,7.610824,7.758269,0.370079,0.019373
4,5,10.9,16.1,10.872322,9.679388,0.477064,-0.109722


In [80]:
# Ens quedem amb els deltas per al dataset
habitatge_final = df_habitatges_metriques[[col for col in df_habitatges_metriques.columns if col.startswith("delta") or col == "codi_barri"]]
resum_estructura(habitatge_final, "habitatge_final")

,tipus,nuls,infinits
columna (habitatge_final),,,
codi_barri,int64,0,0
delta_preu_mitja_m2,float64,0,0
delta_pisos_turistics_1000_hab,float64,0,0


# Combinacio
En aquest apartat combinem els diferents datasets que hem anat creant per crear: 
- Dataset de 2015
- Dataset de 2023
- Dataset amb deltes

In [81]:
# Creem base
barris = dim_barris[["codi_barri"]].copy()

In [85]:
# Conjunt de dades per 2015
df_2015 = barris.merge(poblacio_15, on="codi_barri", how = "left")\
                .merge(econs_15, on = "codi_barri", how = "left")\
                .merge(df_urbanes_15, on = "codi_barri", how = "left")\
                .merge(df_habitatge_15, on = "codi_barri")
resum_estructura(df_2015, "2015")

,tipus,nuls,infinits
columna (2015),,,
codi_barri,int64,0,0
poblacio_total,int64,0,0
pct_pob_estrangera,float64,0,0
pct_pob_estrangera_occidental,float64,0,0
pct_joves,float64,0,0
pct_universitaris,float64,0,0
import_euros,float64,0,0
index_gini,float64,0,0
total_incidents_1000_hab,float64,0,0


In [89]:
# Conjunt de dades per 2015
df_2023 = barris.merge(poblacio_23, on="codi_barri", how = "left")\
                .merge(econs_23, on = "codi_barri", how = "left")\
                .merge(df_urbanes_23, on = "codi_barri", how = "left")\
                .merge(df_habitatge_23, on = "codi_barri")
resum_estructura(df_2023, "2023")

,tipus,nuls,infinits
columna (2023),,,
codi_barri,int64,0,0
poblacio_total,int64,0,0
pct_pob_estrangera,float64,0,0
pct_pob_estrangera_occidental,float64,0,0
pct_joves,float64,0,0
pct_universitaris,float64,0,0
import_euros,float64,0,0
index_gini,float64,0,0
total_incidents_1000_hab,float64,0,0


In [87]:
# Conjunt de dades per 2015
df_deltes = barris.merge(poblacio_final, on="codi_barri", how = "left")\
                .merge(econs_final, on = "codi_barri", how = "left")\
                .merge(urbanes_final, on = "codi_barri", how = "left")\
                .merge(habitatge_final, on = "codi_barri")
resum_estructura(df_deltes, "deltes")

,tipus,nuls,infinits
columna (deltes),,,
codi_barri,int64,0,0
delta_pct_pob_estrangera,float64,0,0
delta_pct_pob_estrangera_occidental,float64,0,0
delta_pct_joves,float64,0,0
delta_pct_universitaris,float64,0,0
delta_poblacio_total,float64,0,0
delta_import_euros,float64,0,0
delta_index_gini,float64,0,0
delta_total_incidents_1000_hab,float64,0,0


In [90]:
# Exportem datasets
df_2015.to_csv("../data/modelling/df_2015.csv", index=False)
df_2023.to_csv("../data/modelling/df_2023.csv", index=False)
df_deltes.to_csv("../data/modelling/df_deltes.csv", index=False)